In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bayes_opt import BayesianOptimization
from catboost import CatBoostRegressor, Pool
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'bayesian_optimization'
OUT.mkdir(parents=True, exist_ok=True)

DATA_PATH = ROOT / 'database.xlsx'
RANDOM_STATE = 2
CV_RANDOM_STATE = 42
SEED = 42
TASK_TYPE = os.getenv('CATBOOST_TASK_TYPE', 'GPU').upper()
CATBOOST_DEVICE_ARGS = {'devices': '0'} if TASK_TYPE == 'GPU' else {}
N_INIT_POINTS = 40
N_ITER = 100
FIGSIZE = (10, 6)
FIG_DPI = 300

FEATURES = ['AM','AMc','Pr','Prc','Sp','Sp2','MSP','PM','CT','Ct','RT','Rt','RH','T','P','GHSV','Inert','H2/CO2']
CATEGORICAL = ['AM','Pr','Sp','Sp2','PM']
CONTINUOUS = [feature for feature in FEATURES if feature not in CATEGORICAL]

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
plt.rcParams['axes.unicode_minus'] = False

data = pd.read_excel(DATA_PATH)
X = data[FEATURES].copy()
y = pd.to_numeric(data['Rate'], errors='coerce')
valid = y.notna()
X, y = X.loc[valid].reset_index(drop=True), y.loc[valid].reset_index(drop=True)
X[CONTINUOUS] = SimpleImputer(strategy='mean').fit_transform(X[CONTINUOUS])
X[CATEGORICAL] = SimpleImputer(strategy='most_frequent').fit_transform(X[CATEGORICAL])
X[CATEGORICAL] = X[CATEGORICAL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL)
test_pool = Pool(X_test, y_test, cat_features=CATEGORICAL)


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=CV_RANDOM_STATE)

def catboost_cv(depth, learning_rate, l2_leaf_reg, bagging_temperature):
    depth = int(round(depth))
    scores = []

    for train_index, valid_index in kf.split(X_train):
        X_fold_train = X_train.iloc[train_index]
        X_fold_valid = X_train.iloc[valid_index]
        y_fold_train = y_train.iloc[train_index]
        y_fold_valid = y_train.iloc[valid_index]

        fold_train_pool = Pool(
            X_fold_train, y_fold_train, cat_features=CATEGORICAL
        )
        fold_valid_pool = Pool(
            X_fold_valid, y_fold_valid, cat_features=CATEGORICAL
        )

        model = CatBoostRegressor(
            iterations=1000,
            depth=depth,
            learning_rate=float(learning_rate),
            l2_leaf_reg=float(l2_leaf_reg),
            bagging_temperature=float(bagging_temperature),
            early_stopping_rounds=100,
            random_seed=42,
            task_type=TASK_TYPE,
            allow_writing_files=False,
            verbose=False,
            **CATBOOST_DEVICE_ARGS,
        )
        model.fit(fold_train_pool, eval_set=fold_valid_pool)
        predictions = model.predict(X_fold_valid)
        scores.append(r2_score(y_fold_valid, predictions))

    return float(np.mean(scores))

pbounds = {
    'depth': (8, 15),
    'learning_rate': (0.2, 0.5),
    'l2_leaf_reg': (1, 10),
    'bagging_temperature': (0, 1),
}

optimizer = BayesianOptimization(
    f=catboost_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=2,
)
optimizer.maximize(init_points=N_INIT_POINTS, n_iter=N_ITER)

In [ ]:
results = pd.DataFrame([
    {**record['params'], 'R2': record['target']}
    for record in optimizer.res
])
results['depth'] = results['depth'].round().astype(int)
results['learning_rate'] = results['learning_rate'].round(3)
results['l2_leaf_reg'] = results['l2_leaf_reg'].round(3)
results['bagging_temperature'] = results['bagging_temperature'].round(3)

best_params = {
    'depth': int(round(optimizer.max['params']['depth'])),
    'learning_rate': round(optimizer.max['params']['learning_rate'], 3),
    'l2_leaf_reg': round(optimizer.max['params']['l2_leaf_reg'], 3),
    'bagging_temperature': round(optimizer.max['params']['bagging_temperature'], 3),
}
best_cv_r2 = optimizer.max['target']

print('Best parameters:', best_params)
print(f'Five-fold CV mean R2: {best_cv_r2:.4f}')

plt.figure(figsize=FIGSIZE, dpi=FIG_DPI)
plt.plot(results['R2'], marker='o', linewidth=1.2, markersize=3)
plt.xlabel('Iteration')
plt.ylabel('Five-fold CV mean R2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
start_time = time.time()

final_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=best_params['learning_rate'],
    depth=best_params['depth'],
    l2_leaf_reg=best_params['l2_leaf_reg'],
    bagging_temperature=best_params['bagging_temperature'],
    eval_metric='RMSE',
    random_seed=42,
    task_type=TASK_TYPE,
    allow_writing_files=False,
    **CATBOOST_DEVICE_ARGS,
)
final_model.fit(train_pool, eval_set=test_pool, verbose=100)

evals_result = final_model.get_evals_result()
training_time = time.time() - start_time
print(f'Training time: {training_time:.2f} seconds')

In [ ]:
train_predictions = final_model.predict(X_train)
test_predictions = final_model.predict(X_test)

train_r2 = r2_score(y_train, train_predictions)
test_r2 = r2_score(y_test, test_predictions)
train_rmse = mean_squared_error(y_train, train_predictions, squared=False)
test_rmse = mean_squared_error(y_test, test_predictions, squared=False)

print(f'Train R2: {train_r2:.4f}')
print(f'Test R2: {test_r2:.4f}')
print(f'Train RMSE: {train_rmse:.4f}')
print(f'Test RMSE: {test_rmse:.4f}')

In [ ]:
result_dir = OUT
results.to_csv(result_dir / 'bayesian_optimization_history.csv', index=False)
with open(result_dir / 'best_params.json', 'w', encoding='utf-8') as file:
    json.dump(best_params, file, ensure_ascii=False, indent=2)
final_model.save_model(result_dir / 'catboost_model.cbm')